In [ ]:
from typing import cast
from matplotlib.figure import Figure
from matplotlib.axes import Axes
from cartopy.mpl.geoaxes import GeoAxes
import sys
from pathlib import Path
import datetime as dt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = "gulf_stream_20240305_20260531"
COLORS = {"cyclone": "tab:blue", "anticyclone": "tab:red"}

In [ ]:
def compute_haversine_km(lon1, lat1, lon2, lat2):
    """Vectorized great-circle distance in km."""
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

## Load data

Number of observations across all eddy polarities across all days, for eddies which have a lifetime of over 60 days (see config)

In [ ]:
frames = []
for polarity in ("cyclone", "anticyclone"):
    pigments_dir = ROOT / "data" / EXPERIMENT / "silver" / "pigments" / polarity
    for fp in sorted(pigments_dir.glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        frames.append(df)

pigments = pd.concat(frames, ignore_index=True)
meta_cols = ["track_id", "date", "pixel_lon", "pixel_lat", "center_lon", "center_lat", "coverage"]
pigment_cols = [c for c in pigments.columns if c not in meta_cols + ["polarity"]]

Pigments: one row per pixel for each eddy on each day

In [ ]:
pigments

In [ ]:
def load_track_props(polarity):
    PET_EPOCH = dt.date(1950, 1, 1)
    zarr_path = ROOT / "data" / EXPERIMENT / "silver" / "eddy_track" / polarity / f"{polarity}_tracks.zarr"
    tracked = TrackEddiesObservations.load_file(str(zarr_path))

    unique_ids = np.unique(tracked.track)
    lifetime = {}
    for tid in unique_ids:
        t = tracked.time[tracked.track == tid]
        lifetime[tid] = int(t.max() - t.min()) + 1

    dates = [PET_EPOCH + dt.timedelta(days=int(d)) for d in tracked.time]
    return pd.DataFrame({
        "track_id": tracked.track.astype(int),
        "date": pd.to_datetime(dates),
        "polarity": polarity,
        "center_lon": (tracked.longitude + 180) % 360 - 180,
        "center_lat": tracked.latitude,
        "radius_km": tracked.radius_e / 1000,
        "amplitude_m": tracked.amplitude,
        "speed_avg": tracked.speed_average,
        "lifetime_days": [lifetime[tid] for tid in tracked.track],
    })

track_properties = pd.concat(
    [load_track_props(p) for p in ("cyclone", "anticyclone")],
    ignore_index=True,
)
print(track_properties.groupby("polarity")["track_id"].nunique())

Track: one row per eddy per day

In [ ]:
track_properties

In [ ]:
pigments = pigments.merge(
    track_properties[["track_id", "date", "polarity",
           "radius_km", "amplitude_m", "speed_avg", "lifetime_days"]],
    on=["track_id", "date", "polarity"],
    how="left",
)

pigments["dist_km"] = compute_haversine_km(
    pigments["pixel_lon"].values, pigments["pixel_lat"].values,
    pigments["center_lon"].values, pigments["center_lat"].values,
)

# Obtain a normalized distance in terms of distance to eddy center
pigments["r_norm"] = pigments["dist_km"] / pigments["radius_km"]


## Overall stats about cyclonic vs anticyclonic eddies

In [ ]:
summary = (
    pigments.groupby("polarity")
    .apply(lambda g: pd.Series({
        "eddies": g["track_id"].nunique(),
        "eddy-dates": g.groupby("track_id")["date"].nunique().sum(),
        "pixels": len(g),
        "median lifetime (d)": g.drop_duplicates("track_id")["lifetime_days"].median(),
        "median radius (km)": g["radius_km"].median(),
        "median amplitude (m)": g["amplitude_m"].median(),
    }), include_groups=False)
)
summary

Distinct eddy-date pairings carried through each stage of the PACE pipeline. Tracked and pigment counts are consistent with the local notebook data; the intermediate `PACE file available` and `collocated RRS` counts come from the PACE ICE audit of the scratch-backed run archive. Bars are stacked by polarity to show both the total cut and the cyclone-anticyclone split.

In [ ]:
stage_counts = pd.DataFrame({
    "stage": [
        "Tracked daily\neddy-dates",
        "PACE file\navailable",
        "Collocated RRS\neddy-dates",
        "Pigment\neddy-dates",
    ],
    "cyclone": [4821, 4749, 514, 509],
    "anticyclone": [3473, 3420, 308, 307],
})
stage_counts["total"] = stage_counts["cyclone"] + stage_counts["anticyclone"]
stage_counts["pct_of_tracked"] = stage_counts["total"] / stage_counts.loc[0, "total"]
stage_counts

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(9, 4.8)),
)
left = np.zeros(len(stage_counts))

for polarity in ("cyclone", "anticyclone"):
    vals = stage_counts[polarity].to_numpy()
    ax.barh(stage_counts["stage"], vals, left=left, color=COLORS[polarity], alpha=0.85, label=polarity)

    for i, (start, val) in enumerate(zip(left, vals)):
        if val >= 250:
            ax.text(start + val / 2, i, f"{val:,}", ha="center", va="center", color="white", fontsize=9, fontweight="bold")

    left = left + vals

for i, (total, pct) in enumerate(zip(stage_counts["total"], stage_counts["pct_of_tracked"])):
    ax.text(total + 150, i, f"{total:,} ({pct:.0%})", va="center", fontsize=9)

ax.set_xlabel("Distinct eddy-date pairings")
ax.set_title("Count of eddy-date pairings through PACE processing")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(0, stage_counts["total"].max() * 1.20)
ax.invert_yaxis()
fig.tight_layout()


In [ ]:
# Obtain mean pigments in an eddy for a particular day
eddy_means = (
    pigments.groupby(["track_id", "date", "polarity"])[pigment_cols]
    .mean()
    .reset_index()
)
eddy_means.loc[eddy_means['track_id'] == 0]

Distribution of per-eddy-date mean T chla for cyclonic (blue) and anticyclonic (red) eddies. Each value is the spatial mean across all PACE pixels within one eddy on one day. Density-normalized histograms allow comparison despite unequal sample sizes between polarities.

In [ ]:
fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(),
)
for pol in ("cyclone", "anticyclone"):
    vals = eddy_means.loc[eddy_means["polarity"] == pol, "T chla"]
    ax.hist(vals, bins=30, alpha=0.5, label=pol, color=COLORS[pol], density=True)
ax.set_xlabel("T chla (eddy mean on a particular day)")
ax.legend()

## Radial pigment profiles

In [ ]:
RADIAL_BINS = [0, 0.25, 0.5, 0.75, 1.0, 1.5]

bin_mids = [(RADIAL_BINS[i] + RADIAL_BINS[i+1]) / 2 for i in range(len(RADIAL_BINS) - 1)]
bin_labels = [f"{RADIAL_BINS[i]:.2f}-{RADIAL_BINS[i+1]:.2f}"
    for i in range(len(RADIAL_BINS) - 1)]

pigments["r_bin"] = pd.cut(pigments["r_norm"], bins=RADIAL_BINS, labels=bin_labels, right=False)
pig_binned = pigments.dropna(subset=["r_bin"])

In [ ]:
def compute_radial_profile(df, col, n_boot=1000):
    grouped = (df.groupby(["track_id", "date", "r_bin"], observed=True)[col]
        .mean().reset_index())

    means, lo, hi = [], [], []
    for label in bin_labels:
        vals = grouped.loc[grouped["r_bin"] == label, col].values
        if len(vals) < 3:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan)
            continue
        res = stats.bootstrap((vals,), np.mean, n_resamples=n_boot,
            confidence_level=0.95, method="percentile")
        means.append(vals.mean())
        lo.append(res.confidence_interval.low)
        hi.append(res.confidence_interval.high)
    return np.array(means), np.array(lo), np.array(hi)

Mean diagnostic-pigment concentration as a function of normalized radial distance (r/R), with 95% bootstrap confidence intervals. Radial bins span 0–0.25, 0.25–0.5, 0.5–0.75, 0.75–1.0, and 1.0–1.5 effective radii. Each bin value averages per-eddy-date bin means, weighting eddies equally.

In [ ]:
DIAGNOSTIC_PIGMENTS = [
    "Fuco", # diatoms
    "HexFuco", # haptophytes
    "Perid", # dinoflagellates
    "Zea", # cyanobacteria
    "DV chla", # prochlorococcus
    "Allo", # cryptophytes
    "MV chlb", # green algae
]

fig, axes = cast(
    tuple[Figure, np.ndarray],
    plt.subplots(2, 4, figsize=(16, 8)),
)
x = np.array(bin_mids)

for i, col in enumerate(DIAGNOSTIC_PIGMENTS):
    ax = axes.flat[i]
    for pol in ("cyclone", "anticyclone"):
        subset = pig_binned.loc[pig_binned["polarity"] == pol]
        m, lo, hi = compute_radial_profile(subset, col)
        ax.plot(x, m, "o-", color=COLORS[pol], label=pol, ms=4)
        ax.fill_between(x, lo, hi, color=COLORS[pol], alpha=0.15)
    ax.set_title(col)
    ax.set_xlabel("r / R")
    ax.set_xlim(0, 1.5)

axes.flat[0].legend()
axes.flat[-1].set_visible(False)
fig.tight_layout()

## Eddy track map

Trajectories of all tracked eddies with lifetime ≥ 60 days in the Gulf Stream region (81°W–56°W, 29°N–44°N), October 2024 – June 2025. Circles mark eddy genesis; squares mark termination. Blue = cyclonic, red = anticyclonic.

In [ ]:
pace_date_counts = (
    pigments.groupby(["polarity", "track_id"])[["date"]]
    .nunique()
    .reset_index().rename(columns={"date": "pace_dates"})
)
top_cyclones = (
    pace_date_counts.loc[pace_date_counts["polarity"] == "cyclone"]
    .sort_values(["pace_dates", "track_id"], ascending=[False, True])
    .head(3)
)
top_anticyclones = (
    pace_date_counts.loc[pace_date_counts["polarity"] == "anticyclone"]
    .sort_values(["pace_dates", "track_id"], ascending=[False, True])
    .head(2)
)
SELECTED_EDDIES = [
    *list(top_cyclones[["polarity", "track_id"]].itertuples(index=False, name=None)),
    *list(top_anticyclones[["polarity", "track_id"]].itertuples(index=False, name=None)),
]
selected_lookup = set(SELECTED_EDDIES)

fig, ax = cast(
    tuple[Figure, GeoAxes],
    plt.subplots(figsize=(10, 7), subplot_kw={"projection": ccrs.PlateCarree()}),
)
ax.set_extent([-81, -56, 29, 44], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#e8e8e8")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

for pol in ("cyclone", "anticyclone"):
    sub = cast(
        pd.DataFrame,
        track_properties.loc[track_properties["polarity"] == pol],
    )
    for tid, grp in sub.sort_values("date").groupby("track_id"):
        is_selected = (pol, int(cast(int, tid))) in selected_lookup
        ax.plot(
            grp["center_lon"],
            grp["center_lat"],
            color=COLORS[pol],
            lw=2.2 if is_selected else 0.7,
            alpha=0.95 if is_selected else 0.18,
            transform=ccrs.PlateCarree(),
            zorder=3 if is_selected else 1,
        )
        if is_selected:
            ax.plot(
                grp["center_lon"].iloc[0],
                grp["center_lat"].iloc[0],
                "o",
                color=COLORS[pol],
                ms=6,
                transform=ccrs.PlateCarree(),
                zorder=4,
            )
            ax.plot(
                grp["center_lon"].iloc[-1],
                grp["center_lat"].iloc[-1],
                "s",
                color=COLORS[pol],
                ms=6,
                transform=ccrs.PlateCarree(),
                zorder=4,
            )

for pol, tid in SELECTED_EDDIES:
    grp = (
        track_properties.loc[
            (track_properties["polarity"] == pol)
            & (track_properties["track_id"] == tid)
        ]
        .sort_values("date")
        .copy()
    )
    label_idx = len(grp) // 2
    ax.text(
        grp["center_lon"].iloc[label_idx],
        grp["center_lat"].iloc[label_idx],
        str(tid),
        transform=ccrs.PlateCarree(),
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            edgecolor=COLORS[pol],
            alpha=0.9,
        ),
        zorder=5,
    )

for pol in ("cyclone", "anticyclone"):
    ax.plot([], [], color=COLORS[pol], lw=2.2, label=pol)

ax.set_title("Best-covered cyclone and anticyclone tracks labeled by id")
ax.legend(loc="upper right")
fig.tight_layout()


![](images/2026-03-25-09-17-08.png)

Open questions
- Plot eddies individually for cyclonic/anticyclonic, see if eddies of the same polarity have a similar shape over their lifetime
    - time on x axis, tchla on y
    - then try normalization based on whole lifetime

In [ ]:
if "SELECTED_EDDIES" not in globals():
    pace_date_counts = (
        pigments.groupby(["polarity", "track_id"])[["date"]]
        .nunique()
        .reset_index().rename(columns={"date": "pace_dates"})
    )
    top_cyclones = (
        pace_date_counts.loc[pace_date_counts["polarity"] == "cyclone"]
        .sort_values(["pace_dates", "track_id"], ascending=[False, True])
        .head(3)
    )
    top_anticyclones = (
        pace_date_counts.loc[pace_date_counts["polarity"] == "anticyclone"]
        .sort_values(["pace_dates", "track_id"], ascending=[False, True])
        .head(2)
    )
    SELECTED_EDDIES = [
        *list(top_cyclones[["polarity", "track_id"]].itertuples(index=False, name=None)),
        *list(top_anticyclones[["polarity", "track_id"]].itertuples(index=False, name=None)),
    ]

eddy_tchla = (
    pigments.groupby(["polarity", "track_id", "date"])[["T chla"]]
    .median()
    .reset_index().rename(columns={"T chla": "tchla_median"})
)

track_windows = (
    cast(
    pd.DataFrame,
    track_properties.groupby(["polarity", "track_id"])["date"]
        .agg(track_start="min", track_end="max")
        .reset_index(),
)
)
track_windows["lifetime_days"] = (
    track_windows["track_end"] - track_windows["track_start"]
).dt.days + 1

selected_meta = pd.DataFrame(
    SELECTED_EDDIES,
    columns=pd.Index(["polarity", "track_id"]),
).merge(track_windows, on=["polarity", "track_id"], how="left")
global_max = (
    eddy_tchla.merge(
        pd.DataFrame(SELECTED_EDDIES, columns=pd.Index(["polarity", "track_id"])),
        on=["polarity", "track_id"],
        how="inner",
    )["tchla_median"].max()
)

fig, axes = cast(
    tuple[Figure, np.ndarray],
    plt.subplots(3, 2, figsize=(14, 10), sharey=True),
)
axes = axes.ravel() # (3, 2) -> (6,)

for idx, ((pol, tid), ax) in enumerate(zip(SELECTED_EDDIES, axes)):
    meta = selected_meta.loc[
        (selected_meta["polarity"] == pol)
        & (selected_meta["track_id"] == tid)
    ].iloc[0]
    grp = (
        eddy_tchla.loc[
            (eddy_tchla["polarity"] == pol)
            & (eddy_tchla["track_id"] == tid)
        ]
        .sort_values("date")
        .copy()
    )

    ax.plot(
        grp["date"],
        grp["tchla_median"],
        "o-",
        color=COLORS[pol],
        lw=1.8,
        ms=4,
    )
    ax.set_xlim(meta["track_start"], meta["track_end"])
    ax.axvline(meta["track_start"], color="0.85", linestyle="--", lw=1)
    ax.axvline(meta["track_end"], color="0.85", linestyle="--", lw=1)
    ax.grid(alpha=0.3)
    ax.set_title(
        f"{pol.capitalize()} {tid}\n"
        f"{len(grp)} PACE dates across {int(meta['lifetime_days'])} track days"
    )
    if len(grp) < 3:
        ax.text(
            0.02,
            0.95,
            "Sparse PACE coverage",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            color="0.35",
        )
    if idx % 2 == 0:
        ax.set_ylabel("T chla (spatial median, ug/L)")
    if idx >= 3:
        ax.set_xlabel("Date")

for ax in axes[len(SELECTED_EDDIES):]:
    ax.remove()

for ax in axes[:len(SELECTED_EDDIES)]:
    ax.set_ylim(0, global_max * 1.05)

fig.suptitle("T chla evolution for eddies with the most PACE dates", y=1.02, fontsize=14)
fig.autofmt_xdate(rotation=30, ha="right")
fig.tight_layout(rect=[0, 0, 1, 0.97])


## Eddy lifetime Tchla trajectories

Per-eddy T chla trajectories plotted against normalized eddy lifetime (0 = first observation, 1 = last). Each thin line traces one eddy's spatial median T chla across PACE observation dates. Only eddies with lifetime ≥ 60 days are included; gaps correspond to dates without valid PACE retrievals.

In [ ]:
# Per-eddy-date spatial median Tchla
eddy_medians = (
    cast(
    pd.DataFrame,
    pigments.groupby(["track_id", "date", "polarity"])["T chla"]
        .median()
        .reset_index(),
)
)

# Normalized eddy age: 0 = first observation, 1 = last
date_range = (
    cast(
    pd.DataFrame,
    cast(
            pd.DataFrame,
            eddy_medians.groupby(["track_id", "polarity"])["date"].agg(["min", "max"]),
        )
        .rename(columns={"min": "first_date", "max": "last_date"})
        .reset_index(),
)
)
eddy_medians = cast(
    pd.DataFrame,
    eddy_medians.merge(date_range, on=["track_id", "polarity"]),
)

span = (eddy_medians["last_date"] - eddy_medians["first_date"]).dt.total_seconds()
elapsed = (eddy_medians["date"] - eddy_medians["first_date"]).dt.total_seconds()
eddy_medians["age_frac"] = np.where(span > 0, elapsed / span, 0.0)

fig, axes = cast(
    tuple[Figure, np.ndarray],
    plt.subplots(1, 2, figsize=(14, 5), sharey=True),
)

for ax, pol in zip(axes, ("cyclone", "anticyclone")):
    sub = cast(
        pd.DataFrame,
        eddy_medians.loc[eddy_medians["polarity"] == pol],
    )

    for tid, grp in sub.sort_values("age_frac").groupby("track_id"):
        ax.plot(grp["age_frac"], grp["T chla"],
            color=COLORS[pol], alpha=0.25, lw=0.8)

    ax.set_title(f"{pol.capitalize()} (n={sub['track_id'].nunique()})")
    ax.set_xlabel("Fraction of observed lifetime")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.5)

axes[0].set_ylabel("T chla (spatial median, µg/L)")
fig.suptitle("Eddy lifetime Tchla trajectories", y=1.02, fontsize=13)
fig.tight_layout()

Figure out why there's sudden jumps and fewer points than we think should be there.

Pick out a smaller number of eddies.



## Daily concentration time series

Daily mean T chla concentration per polarity, averaged across all eddies observed on each date. Two-level averaging (pixel → eddy mean, then eddy means → daily mean) ensures each eddy contributes equally regardless of size. Blue = cyclonic, red = anticyclonic.

In [ ]:
# Daily mean T chla (mean of per-eddy means, already in eddy_means)
daily_tchla = (
    eddy_means.groupby(["date", "polarity"])["T chla"]
    .mean()
    .reset_index()
)

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(10, 4)),
)
for pol in ("cyclone", "anticyclone"):
    color = "tab:blue" if pol == "cyclone" else "tab:red"
    sub = cast(
        pd.DataFrame,
        daily_tchla.loc[daily_tchla["polarity"] == pol].sort_values("date"),
    )
    ax.plot(sub["date"], sub["T chla"], color=color, label=pol, lw=1)
ax.set_title("T chla")
ax.set_ylabel("mg/m³")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
